# Session 9 — Wasserstein Distances

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S8 turned the optimal-transport problem into a tractable LP. S9 turns its *cost* into
a genuine **metric** — the **Wasserstein distance**. It has three faces we will meet:

1. it satisfies the **metric axioms** (identity of indiscernibles, symmetry, triangle inequality);
2. in **one dimension** it admits a stunning **closed form** — *just sort and match by quantile* — reducing the LP to a sort, $\mathcal{O}(n^3) \to \mathcal{O}(n\log n)$;
3. it is the *transport-side* geometry whose **geodesics are the McCann interpolations** we already met informally in S6.

This is also the right place to finally state quantitatively *why* Wasserstein
respects geometry where KL does not.

## 0. What you'll be able to do

- State the **Wasserstein-$p$ distance** $W_p(\mu, \nu)$ and verify its **metric axioms**.
- Compute $W_p$ on discrete data via the **Kantorovich LP** (S8's machinery).
- In one dimension, compute $W_p$ exactly via the **quantile (sort-and-match) closed form**, and **verify it agrees with the LP**.
- See **$W_1$ as the area between CDFs** — the cleanest geometric reading.
- Compute the **McCann displacement geodesic** at any time $t$, with formal definition.
- Watch quantitatively how **$W_2$ tracks displacement linearly** while **KL diverges** as supports separate.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.geometry.info_geometry import mixture_interpolation
from qot_course.infotheory.classical import kl_divergence
from qot_course.transport.discrete import squared_euclidean_cost, cityblock_cost
from qot_course.transport.wasserstein import (
    wasserstein_distance, wasserstein_1d, mccann_geodesic_atoms_1d,
)

np.random.seed(42)
viz.use_course_style()

## 1. The Wasserstein-$p$ distance — definition

Take two probability measures $\mu, \nu$ on $\mathbb{R}^d$ and the Euclidean ground cost
$c(x, y) = \|x - y\|$. The **Wasserstein-$p$ distance** ($p \ge 1$) is

$$
\boxed{\;W_p(\mu, \nu) = \Bigl(\inf_{\pi \in \Pi(\mu, \nu)} \int \|x - y\|^p\,\mathrm{d}\pi(x, y)\Bigr)^{1/p}\;}
$$

where $\Pi(\mu, \nu)$ is the set of couplings (joint distributions with marginals
$\mu, \nu$) — i.e. the transportation polytope of S8. The infimum **is attained**
(the polytope is compact and the integral is continuous), so we can write $W_p$ as a
function of the optimal coupling $\pi^\star$.

**Metric axioms.** $W_p$ is a *metric* on the Wasserstein space of distributions of
finite $p$-th moment (Villani, 2003, Thm.~7.3): (i) $W_p(\mu, \nu) \ge 0$ and $= 0$ iff
$\mu = \nu$; (ii) $W_p(\mu, \nu) = W_p(\nu, \mu)$; (iii) the **triangle inequality**
$W_p(\mu, \rho) \le W_p(\mu, \nu) + W_p(\nu, \rho)$ — proven via the *glueing lemma*
on couplings. Let's verify all three numerically.

In [ ]:
mu  = (np.array([0.0, 1.0, 2.0]), np.array([0.2, 0.3, 0.5]))
nu  = (np.array([0.5, 1.5, 2.5]), np.array([0.4, 0.4, 0.2]))
rho = (np.array([-0.5, 0.0, 3.0]), np.array([0.1, 0.5, 0.4]))

d_mu_mu  = wasserstein_1d(*mu,  *mu,  p=2)
d_mu_nu  = wasserstein_1d(*mu,  *nu,  p=2)
d_nu_mu  = wasserstein_1d(*nu,  *mu,  p=2)
d_mu_rho = wasserstein_1d(*mu,  *rho, p=2)
d_nu_rho = wasserstein_1d(*nu,  *rho, p=2)

print(f"identity:   W_2(mu, mu) = {d_mu_mu:.6f}                            -> 0")
print(f"symmetry:   W_2(mu, nu) = {d_mu_nu:.4f}   W_2(nu, mu) = {d_nu_mu:.4f}")
print(f"triangle:   W_2(mu, rho) = {d_mu_rho:.4f}  <=  W_2(mu, nu) + W_2(nu, rho) = {d_mu_nu + d_nu_rho:.4f}")

**Read the output.** All three axioms hold: $d(\mu,\mu)=0$ to machine precision,
$d(\mu,\nu)=d(\nu,\mu)$, and the route via $\nu$ is no shorter than the direct path.
Triangle inequalities like this one are why Wasserstein is a true geometry on
distributions — unlike KL, which is asymmetric and fails the triangle inequality.

## 2. The 1-D closed form — sort and match by quantile

In one dimension, the LP **collapses**. Let $F_\mu$ and $F_\nu$ be the cumulative
distribution functions of $\mu$ and $\nu$. The unique optimal transport map is the
**quantile composition** (Brenier; Villani, 2003, Thm.~2.18):

$$ T = F_\nu^{-1} \circ F_\mu, $$

i.e. *every unit of $\mu$-mass at quantile $u$ is sent to the $\nu$-position with the
same quantile $u$*. The cost integrates to

$$
\boxed{\;W_p^p(\mu, \nu) = \int_0^1 \bigl|F_\mu^{-1}(u) - F_\nu^{-1}(u)\bigr|^p\,\mathrm{d}u.\;}
$$

For discrete atomic distributions this is a **finite sum** over the *common refinement*
of the two CDFs --- computable in $\mathcal{O}((n+m) \log(n+m))$ instead of the
$\mathcal{O}(n^3)$ network simplex. Let's verify the formula numerically.

In [ ]:
rng = np.random.default_rng(11)
n, m = 6, 8
pa = np.sort(rng.uniform(-3.0, 3.0, size=n))
pb = np.sort(rng.uniform(-3.0, 3.0, size=m))
wa = rng.random(n); wa /= wa.sum()
wb = rng.random(m); wb /= wb.sum()

C = squared_euclidean_cost(pa, pb)
w2_lp        = wasserstein_distance(wa, wb, C, p=2)
w2_quantile  = wasserstein_1d(pa, wa, pb, wb, p=2)

print(f"W_2 via Kantorovich LP (POT network simplex) = {w2_lp:.8f}")
print(f"W_2 via 1-D quantile closed form             = {w2_quantile:.8f}")
print(f"agreement?  {np.isclose(w2_lp, w2_quantile)}")

**Read the output.** The two methods agree to machine precision. The closed form
needed only a sort plus a scan of CDF breakpoints; the LP needed a full network
simplex iteration. In 1-D the LP is *overkill* --- which is exactly why we will need
clever algorithms (Sinkhorn in S10) to make OT practical in higher dimensions, where
this gift evaporates.

### A clean special case: $W_1$ as the area between CDFs

For $p = 1$ specifically, the closed form admits a beautiful **horizontal** reading
(Vallender, 1974):

$$ W_1(\mu, \nu) = \int_{-\infty}^{+\infty} \bigl|F_\mu(x) - F_\nu(x)\bigr|\,\mathrm{d}x. $$

That is: the **area between the two CDFs**. Watch.

In [ ]:
# Two smooth bumps on a fine grid (continuous-like discretisation).
grid = np.linspace(-1.0, 11.0, 600)
def density(center, sigma):
    pdf = np.exp(-0.5 * ((grid - center) / sigma) ** 2)
    return pdf / np.trapezoid(pdf, grid)   # density (integrates to 1)

pdf_mu = density(3.0, 0.8)
pdf_nu = density(7.0, 1.2)

# Convert to discrete masses on the grid (one atom per bin).
dx = grid[1] - grid[0]
mass_mu = pdf_mu * dx; mass_mu /= mass_mu.sum()
mass_nu = pdf_nu * dx; mass_nu /= mass_nu.sum()

cdf_mu = np.cumsum(mass_mu)
cdf_nu = np.cumsum(mass_nu)
# Riemann sum approximation of the integral int |F_mu - F_nu| dx:  sum(|...|) * dx.
w1_via_area    = float(np.sum(np.abs(cdf_mu - cdf_nu)) * dx)
w1_via_closed_form = wasserstein_1d(grid, mass_mu, grid, mass_nu, p=1)

print(f"W_1 via 'area between CDFs'                       = {w1_via_area:.4f}")
print(f"W_1 via 1-D quantile closed form (same thing)     = {w1_via_closed_form:.4f}")
print(f"Expected ≈ 4.0   (the bump centres are 4 apart)")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(grid, pdf_mu, color=viz.SOURCE_COLOR, lw=2, label=r"$\mu$ at $x=3$")
axes[0].plot(grid, pdf_nu, color=viz.TARGET_COLOR, lw=2, label=r"$\nu$ at $x=7$")
axes[0].set_ylabel("density"); axes[0].set_title("Two distributions on a line", pad=10)
axes[0].legend()

axes[1].plot(grid, cdf_mu, color=viz.SOURCE_COLOR, lw=2, label=r"$F_\mu$")
axes[1].plot(grid, cdf_nu, color=viz.TARGET_COLOR, lw=2, label=r"$F_\nu$")
axes[1].fill_between(grid, cdf_mu, cdf_nu,
                     color=viz.FLOW_COLOR, alpha=0.32,
                     label=r"$W_1 = \int |F_\mu - F_\nu|\,\mathrm{d}x$")
axes[1].set_xlabel("position  $x$"); axes[1].set_ylabel("cumulative probability")
axes[1].set_title(f"$W_1$ is the area between the two CDFs  ≈ {w1_via_area:.3f}", pad=10)
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Read both panels.** The **top** shows two well-separated bumps; the **bottom** shows
their CDFs --- a smooth shifted pair. The **green shaded region** between them is
precisely the $W_1$ distance, and its area numerically agrees with the closed-form
$W_1 \approx 4.0$ (the bump centres are 4 apart). Same geometric content, two readings
of the same integral.

## 3. McCann displacement interpolation — the W₂ geodesic

The Wasserstein-2 distance is more than a metric: $(W_2, \mathcal{P}_2)$ is a
**Riemannian-like geodesic space** (Otto, 2001). Its **geodesic** between $\mu$ and
$\nu$ is the **McCann interpolation**:

$$ \mu_t = \bigl((1 - t)\,\mathrm{Id} + t\,T\bigr)_{\#}\,\mu, \qquad t \in [0, 1], $$

where $T = F_\nu^{-1} \circ F_\mu$ is the optimal map. Each unit of mass slides
linearly from its $\mu$-position to its $\nu$-position. We compute the geodesic at
explicit time snapshots and watch the peak *move*.

In [ ]:
# Use the same two bumps as above (as atomic masses on the fine grid).
times = [0.0, 0.25, 0.5, 0.75, 1.0]

# McCann atoms at each time (the actual W_2 geodesic).
mccann_snapshots = [
    mccann_geodesic_atoms_1d(grid, mass_mu, grid, mass_nu, t) for t in times
]
# For comparison, the *mixture* / KL-side interpolation (which is bimodal at t=0.5).
mixture_snapshots = [mixture_interpolation(mass_mu, mass_nu, t) for t in times]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
cmap = plt.colormaps["viridis"]

for k, ((positions_t, weights_t), t) in enumerate(zip(mccann_snapshots, times)):
    color = cmap(k / (len(times) - 1))
    # Rebin atomic representation onto the grid for plotting.
    binned = np.zeros_like(grid)
    edges = np.concatenate([[grid[0] - dx/2],
                            0.5 * (grid[:-1] + grid[1:]),
                            [grid[-1] + dx/2]])
    counts, _ = np.histogram(positions_t, bins=edges, weights=weights_t)
    axes[0].plot(grid, counts, color=color, lw=2, label=f"$t={t:.2f}$")

axes[0].set_ylabel("mass"); axes[0].set_title("McCann geodesic ($W_2$): the peak slides", pad=10)
axes[0].legend(loc="upper right", ncol=5, fontsize=9)

for k, (pt, t) in enumerate(zip(mixture_snapshots, times)):
    color = cmap(k / (len(times) - 1))
    axes[1].plot(grid, pt, color=color, lw=2, label=f"$t={t:.2f}$")
axes[1].set_xlabel("position  $x$"); axes[1].set_ylabel("mass")
axes[1].set_title("Mixture interpolation (information side): bimodal at $t=0.5$", pad=10)
axes[1].legend(loc="upper right", ncol=5, fontsize=9)

plt.tight_layout()
plt.show()

**Read both panels.**

- **Top — the $W_2$ McCann geodesic.** The single bump *slides* smoothly from $x = 3$ to $x = 7$, keeping a single mode throughout. The transport geometry recognises that the bins live on a line and that there is empty space between the two original bumps; it chooses to *carry the mass through* that space.
- **Bottom — the mixture (information side).** Two static peaks at $x = 3$ and $x = 7$, their amplitudes morphing. At $t = 0.5$ the distribution is **bimodal** — the same picture we saw informally in S6.

This is exactly the *two-geometries-on-one-simplex* punchline of S6, now backed by the
formal $W_2$ geodesic.

## 4. Why $W_p$ respects geometry — quantitative demo

The cleanest demonstration: take a single bump $\mu$ and slide it by a distance $d$.

$$
W_p(\mu, \mu_{\text{shifted by } d}) = d
\qquad \text{but} \qquad
D_{\mathrm{KL}}(\mu \,\|\, \mu_{\text{shifted by } d}) \to +\infty
$$

as $d$ grows enough for the supports to (numerically) separate. Wasserstein **measures
the distance the mass must move**; KL is **blind** to ground geometry and gets dominated
by tail mismatches.

In [ ]:
shifts = np.arange(0.0, 7.5, 0.5)
ref = density(3.0, 0.8)
ref_mass = ref * dx; ref_mass /= ref_mass.sum()

w2_curve, kl_curve = [], []
for d in shifts:
    shifted = density(3.0 + d, 0.8)
    shifted_mass = shifted * dx; shifted_mass /= shifted_mass.sum()
    w2_curve.append(wasserstein_1d(grid, ref_mass, grid, shifted_mass, p=2))
    # KL is computed in bits; clip the support away from zero defensively.
    kl_curve.append(kl_divergence(ref_mass + 1e-300, shifted_mass + 1e-300))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax_kl = ax.twinx()
l1, = ax.plot(shifts, w2_curve, "-o", color=viz.SOURCE_COLOR, lw=2, label=r"$W_2(\mu, \mu_d)$  (left axis)")
l2, = ax_kl.plot(shifts, kl_curve, "-s", color=viz.TARGET_COLOR, lw=2, label=r"$D_{\mathrm{KL}}(\mu \,\|\, \mu_d)$ (right axis, log)")
ax_kl.set_yscale("log")
ax.set_xlabel("displacement  $d$")
ax.set_ylabel(r"$W_2$  (linear)", color=viz.SOURCE_COLOR)
ax_kl.set_ylabel(r"$D_{\mathrm{KL}}$ (log scale)", color=viz.TARGET_COLOR)
ax.set_title("$W_2$ grows linearly in displacement; KL blows up as supports separate", pad=12)
ax.legend(handles=[l1, l2], loc="upper left")
plt.tight_layout()
plt.show()

**Read the figure.** The **violet $W_2$ curve** is a straight line through the origin
with slope $1$ — exactly $W_2(\mu, \mu_d) = d$ (a hallmark of the transport geometry).
The **amber KL curve** (right axis, log scale) climbs *exponentially* once the
displaced bump leaves the support of the reference: for $d \gtrsim 3$ the two bumps
share essentially no probability mass and KL diverges. **Same data, two distances, two
universes**. This is the quantitative version of the S6 mixture-vs-Wasserstein image,
and the reason why OT is the right metric whenever the ground space carries geometry —
images, signals, neural recordings, density matrices.

## 5. Dictionary update — the transport row, formalised

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ on positions $\{x_i\}$ | density matrix $\rho$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ |
| transport plan / coupling $P$ | quantum coupling: bipartite $\rho_{AB}$ *(S13)* |
| Kantorovich LP $\min \langle C, P\rangle$ | quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$ *(S13)* |
| **Wasserstein-$p$ distance $W_p(\mu, \nu)$** | **quantum Wasserstein** *(S13–S14)* |
| **1-D closed form $W_p^p = \int |F_\mu^{-1} - F_\nu^{-1}|^p$** | **Gaussian closed form (Bures–Wasserstein, S11)** |
| **McCann geodesic $\mu_t$** | **quantum displacement interpolation** *(S14)* |

## 6. Your turn

1. **The Vallender identity.** Verify analytically that for two atomic distributions on
   the integers, the area-between-CDFs formula $W_1 = \sum_x |F_\mu(x) - F_\nu(x)|$ agrees
   with the closed form $W_1 = \int_0^1 |F_\mu^{-1} - F_\nu^{-1}|\,\mathrm{d}u$. *Hint:
   integrate by parts on the right-hand side.*
2. **$W_2$ between Gaussians on a fixed mean.** Take $\mu = \mathcal{N}(0, \sigma_\mu^2)$
   and $\nu = \mathcal{N}(0, \sigma_\nu^2)$, both centred at $0$. Compute $W_2$
   numerically (discretise on a grid) and compare with the closed form
   $W_2(\mu, \nu) = |\sigma_\mu - \sigma_\nu|$ (a 1-D special case of the
   **Bures–Wasserstein** formula we will lift to density matrices in S11).
3. **A bimodal challenge.** Take $\mu = \tfrac{1}{2}\delta_{-1} + \tfrac{1}{2}\delta_{+1}$
   and $\nu = \delta_{0}$ (a Dirac at the origin). Compute $W_1$, $W_2$, and the McCann
   midpoint $\mu_{1/2}$. Is the midpoint bimodal, unimodal, or something else?

## Conclusions & references

- $W_p$ is a genuine **metric** on the space of distributions of finite $p$-th moment.
- In **1-D** it has the explicit **quantile closed form** — *sort and match*. The LP
  collapses to an $\mathcal{O}((n+m)\log(n+m))$ sort.
- $W_1$ equals the **area between the two CDFs** (Vallender, 1974).
- The **McCann geodesic** is the actual $W_2$-geodesic: it slides the mass through the
  ground space rather than morphing amplitudes.
- $W_2$ tracks **displacement linearly**; KL is dominated by tail mismatches and
  diverges when supports separate. This is why we need transport.
- **Next (S10 — Duality & Sinkhorn):** the dual view of the LP; entropic regularisation;
  and Amari's bridge — *Sinkhorn is where Wasserstein meets KL*.

**References:** C. Villani, *Topics in Optimal Transportation* (AMS, 2003), ch. 2;
G. Peyré & M. Cuturi, *Computational Optimal Transport* (NoW, 2019), ch. 2; R. McCann,
"A convexity principle for interacting gases", Adv. Math. 128, 153 (1997);
S. S. Vallender, "Calculation of the Wasserstein distance between probability
distributions on the line", Theory Probab. Appl. 18, 784 (1974); F. Otto, "The geometry
of dissipative evolution equations: the porous medium equation", Comm. PDE 26, 101
(2001). Previous: `notebooks/s08_monge_kantorovich.ipynb`.